# iRED S² and psi circular variance by secondary structure × exposure (PANAV, 10-seed ensemble)

The panel cell filters residues by SS and burial persistence (`PERSIST_CUT`).

In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, wilcoxon, pearsonr, spearmanr
import warnings
warnings.filterwarnings("ignore")


def set_style(base=13, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base,
        "axes.titlesize": base,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "svg.fonttype": "none",
    })


set_style()
BLUE, RED, GREEN, GREY, PURPLE = "#61AFEF", "#E06C75", "#98C379", "#999999", "#C678DD"


def style_axes(ax):
    ax.grid(alpha=.3, ls="--")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


def save(fig, name):
    fig.savefig(f"{FIG}/{name}.svg", bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.png", dpi=200, bbox_inches="tight")
    print("saved", name)


S2PSI_JSON = ("../../data/analysis/Fig4/per_protein_analysis/md_1us_combined_48_justs2_justdihedral_panav10seedens.json")

FIG = "Figs"

SS_ORDER = ["helix", "sheet", "coil"]
EXPOSURES = ["buried", "exposed"]
COL = {"buried": "#61AFEF", "exposed": "#E06C75"}
PREDICTORS = ["s2_ired", "psi_var_self", "psi_var_prev", "psi_var_combined"]

data = json.load(open(S2PSI_JSON))["proteins"]

rows = []
for entry, v in data.items():
    pr = v["per_residue"]
    n = len(pr["resnum"])
    for i in range(n):
        rows.append(dict(
            entry=entry, resnum=pr["resnum"][i], aa=pr["aa"][i],
            ss=pr["ss_class"][i], exposure=pr["exposure"][i],
            err=pr["acerr"][i],
            s2_ired=pr["s2_ired"][i],
            psi_var_self=pr["psi_var_self"][i],
            psi_var_prev=pr["psi_var_prev"][i],
            psi_var_combined=pr["psi_var_combined"][i],
        ))

A = pd.DataFrame(rows)
print(f"{len(A)} residues over {A.entry.nunique()} proteins (pre-persist90)")

## Group summary by SS × exposure

In [ ]:
def report(D, tag):
    print("=" * 100)
    print(f"{tag}   (n_residues={len(D)}, n_proteins={D.entry.nunique()})")
    print("=" * 100)

    def _summarize(g):
        out = {"n": len(g), "n_prot": g["entry"].nunique()}
        for m in PREDICTORS:
            out[m] = g[m].mean()
            out[f"{m}_std"] = g[m].std()
        return pd.Series(out)

    summary = D.groupby(["ss", "exposure"]).apply(_summarize)
    print(summary.round(3).to_string())
    print("  (s2_ired: HIGH = more rigid.  psi_var_*: HIGH = more dynamic (circular variance of psi).)")
    print("  (*_std = sample std within the group, i.e. across residues -- NOT the SEM of the group mean)")
    return summary


summary_all = report(A, "ALL residues (persist90-restricted)")

## Per-entry correlation with |N error|

In [ ]:
MIN_RES = 5
EXPOSURE_ORDER = ["buried", "exposed"]


def pool_by_entry(sub, xcol):
    """One row per entry: mean predictor + median err (err is heavy-tailed, hence the median)."""
    d = sub[[xcol, "err", "entry"]].dropna()
    g = d.groupby("entry").agg(x=(xcol, "mean"), err=("err", "median"), n=("err", "size"))
    return g[g.n >= MIN_RES]


def corr(g):
    if len(g) < 5:
        return dict(n_prot=len(g), rho=np.nan, rp=np.nan, r=np.nan, pp=np.nan)
    rho, rp = spearmanr(g.x, g.err)
    r, pp = pearsonr(g.x, g.err)
    return dict(n_prot=len(g), rho=rho, rp=rp, r=r, pp=pp)


for xcol in PREDICTORS:
    print("=" * 88)
    print(f"per-entry pooled  {xcol}  vs  err     (one point per protein: mean {xcol} vs median err)")
    print("=" * 88)
    print(f"{'stratum':22s} {'n_prot':>7s} {'med_res':>8s} | {'Spearman':>9s} {'p':>10s} | "
          f"{'Pearson':>8s} {'p':>10s}")
    print("-" * 88)
    for exp in EXPOSURE_ORDER:
        for ss in SS_ORDER:
            g = pool_by_entry(A[(A.exposure == exp) & (A.ss == ss)], xcol)
            c = corr(g)
            med_res = int(g.n.median()) if len(g) else 0
            print(f"{exp + '/' + ss:22s} {c['n_prot']:7d} {med_res:8d} | "
                  f"{c['rho']:+9.3f} {c['rp']:10.2e} | {c['r']:+8.3f} {c['pp']:10.2e}")
        g = pool_by_entry(A[A.exposure == exp], xcol)
        c = corr(g)
        print(f"{exp + ' (all ss)':22s} {c['n_prot']:7d} {int(g.n.median()):8d} | "
              f"{c['rho']:+9.3f} {c['rp']:10.2e} | {c['r']:+8.3f} {c['pp']:10.2e}")
        print("-" * 88)
    g = pool_by_entry(A, xcol)
    c = corr(g)
    print(f"{'ALL':22s} {c['n_prot']:7d} {int(g.n.median()):8d} | "
          f"{c['rho']:+9.3f} {c['rp']:10.2e} | {c['r']:+8.3f} {c['pp']:10.2e}")
    print()

In [ ]:
# ================= cross-model robustness, exposed coil: per-protein MD predictor vs |N error| =================
# Self-contained. For each model, |error| is recomputed from its own 10-seed-ensemble predictions with
# the same gate as the JSON's acerr (Pro / None / prediction mask); the MD predictors come from the JSON
# and are model-independent. Per protein (same as pool_by_entry): mean predictor vs median |error|,
# proteins with < MIN_RES residues in the stratum dropped. Bar = Spearman rho, whisker = 95% CI from a
# protein bootstrap, text = Spearman p. One figure per predictor, shared y-range across the three.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

FIG4 = "../../data/analysis/Fig4"
FIG = "Figs"
S2PSI_JSON = f"{FIG4}/per_protein_analysis/md_1us_combined_48_justs2_justdihedral_panav10seedens.json"
GT = "../../data/rBMRB/cluster_30p/PANAV/test.json"
MODELS = {                      # label -> prediction-file suffix; order = bar order
    "SLAE-CS":       "",
    "ESM3\n(str + seq)":          "_esm3",
    "MACE-OFF23":          "_mace",
    "ESM3\n(seq only)": "_esm3_seqonly",
    "ProteinMPNN\n(finetuned)":       "_mpnn_ft",
    # "MPNN frozen":   "_mpnn_frozen",
}
MODEL_C = {"SLAE-CS": "#3F729B", "ESM3\n(str + seq)": "#A9CCE3", "ESM3\n(seq only)": "#A9CCE3",
           "MACE-OFF23": "#A9CCE3", "ProteinMPNN\n(finetuned)": "#A9CCE3", "ProteinMPNN\n(frozen)": "#6E6E6E"}
STRATUM = ("exposed", "coil")
XCOLS = {                       # predictor -> axis label (psi variance: HIGH = dynamic; S2: HIGH = rigid)
    "psi_var_self": "mean ψ(i) variance",
    "psi_var_prev": "mean ψ(i−1) variance",
    "s2_ired":      "mean S$^{2}$(iRED)",
}
MIN_RES = 5
B_BOOT = 2000


def set_style(base=13, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base, "axes.labelsize": base,
        "xtick.labelsize": base - 3, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",
    })


# ---------- per-residue table: MD predictor + gate from the JSON ----------
data = json.load(open(S2PSI_JSON))["proteins"]
A = pd.DataFrame([dict(entry=e, resnum=p["per_residue"]["resnum"][i], ss=p["per_residue"]["ss_class"][i],
                       exposure=p["per_residue"]["exposure"][i], gated=p["per_residue"]["acerr"][i] is not None,
                       **{c: p["per_residue"][c][i] for c in XCOLS})
                  for e, p in data.items() for i in range(len(p["per_residue"]["resnum"]))])

PERSIST_CUT = 0.5               # None = no persist filter; 0.5 / 0.9 = both ss and burial persistence >= cut
P = pd.read_csv(f"{FIG4}/per_protein_analysis/persist90_ss_burial_simple.csv")[["entry", "resnum", "ss_persist", "burial_persist"]]
A = A.merge(P, on=["entry", "resnum"], how="left")
if PERSIST_CUT is not None:
    A = A[(A.ss_persist >= PERSIST_CUT) & (A.burial_persist >= PERSIST_CUT)].reset_index(drop=True)

# ---------- |error| per model, same recipe as the JSON's acerr ----------
gt = json.load(open(GT))
inv = {v: k for k, v in gt["entry_ID"].items()}
for label, tag in MODELS.items():
    pred = json.load(open(f"{FIG4}/results/predictions_N_seed_ensemble_10seed{tag}.json"))
    err = {}
    for e in A.entry.unique():
        seq, shifts = gt["sequence"][inv[e]], gt["N"][inv[e]]
        n = len(seq)
        pa = np.array(pred["mean"][e])[:n]
        pm = np.array(pred["mask"][e], dtype=bool)[:n]
        for i in range(n):
            if seq[i] != "P" and shifts[i] is not None and pm[i]:
                err[(e, i + 1)] = abs(float(pa[i]) - float(shifts[i]))
    A[label] = [err.get((e, r), np.nan) if g else np.nan for e, r, g in zip(A.entry, A.resnum, A.gated)]
print("residues with |error| per model:", {m: int(A[m].notna().sum()) for m in MODELS})

# ---------- per-protein Spearman + protein bootstrap CI (exposed coil) ----------
rng = np.random.default_rng(0)
exp, ss = STRATUM
sub = A[(A.exposure == exp) & (A.ss == ss)]
rows = []
for xcol in XCOLS:
    for m in MODELS:
        g = (sub[["entry", xcol, m]].dropna()
             .groupby("entry").agg(x=(xcol, "mean"), err=(m, "median"), n=(m, "size")))
        g = g[g.n >= MIN_RES]
        rho, p = spearmanr(g.x, g.err)
        x, y = g.x.to_numpy(), g.err.to_numpy()
        boot = [spearmanr(x[i], y[i])[0] for i in (rng.integers(0, len(g), len(g)) for _ in range(B_BOOT))]
        lo, hi = np.nanpercentile(boot, [2.5, 97.5])
        rows.append(dict(predictor=xcol, model=m, n_prot=len(g), med_res=int(g.n.median()),
                         rho=rho, ci_lo=lo, ci_hi=hi, p=p))
RR = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(f"{exp} {ss}: per protein, mean predictor vs median |error|")
print(RR.round(3).to_string(index=False))


In [ ]:
# ---------- one bar chart per predictor, one bar per model ----------
# Enable one predictor in XCOLS at a time and set the matching y-range below; figsize stays (7.5, 5.6).
#   psi_var_self: ax.set_ylim(-1, 0.35)     SI Fig S10a
#   s2_ired:      ax.set_ylim(-0.35, 1)     SI Fig S10c (rho is positive: high S2 = rigid)
#   psi_var_prev: ax.set_ylim(-1, 0.35)     not used in the paper
set_style(base=16)
Y_LO = min(RR.ci_lo.min(), 0) - 0.16
Y_HI = max(RR.ci_hi.max(), 0) + 0.08
xs = np.arange(len(MODELS))
XCOLS = {                       # predictor -> axis label (psi variance: HIGH = dynamic; S2: HIGH = rigid)
    "psi_var_self": "mean ψ(i) variance",
    # "s2_ired":      "mean S$^{2}$(iRED)",
}
for xcol, xlabel in XCOLS.items():
    R = RR[RR.predictor == xcol].set_index("model").loc[list(MODELS)]
    fig, ax = plt.subplots(figsize=(7.5, 5.6))
    ax.bar(xs, R.rho, 0.7, color=[MODEL_C[m] for m in MODELS], zorder=2)
    ax.errorbar(xs, R.rho, yerr=[R.rho - R.ci_lo, R.ci_hi - R.rho], fmt="none",
                ecolor="#444", elinewidth=1.2, capsize=4, zorder=3)
    # for x, (m, r) in zip(xs, R.iterrows()):
    #     sig = r.p < 0.05
    #     ax.text(x, Y_LO + 0.02, f"p={r.p:.1g}" if r.p >= 1e-3 else f"p={r.p:.0e}", ha="center", va="bottom",
    #             fontsize=14, fontweight="bold" if sig else "normal", color="k" if sig else "#777", font="Arial")
    def stars(p):
        return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "n.s."

    for x, (m, r) in zip(xs, R.iterrows()):
        sig = r.p < 0.05
        ax.text(x, Y_LO + 0.02, stars(r.p), ha="center", va="bottom",
                fontsize=17, fontweight="bold" if sig else "normal", color="k" if sig else "#777", font="Arial")
    ax.axhline(0, color="k", lw=1)
    ax.set_xticks(xs)
    ax.set_xticklabels(list(MODELS))
    ax.set_ylim(-1, 0.35)
    ax.set_ylabel("Spearman ρ (per protein)")
    ax.set_title(f"Exposed coil: {xlabel} vs median |error|",
                 fontsize=14, font="Arial")
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)
    plt.tight_layout()
    name = f"s2psi_{xcol.replace('_', '')}_perprotein_rho_models_exposedcoil_panav10seedens"
    fig.savefig(f"{FIG}/{name}.svg", bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.png", dpi=300, bbox_inches="tight")
    print("saved", name)
    plt.show()

In [ ]:
# ================= SLAE-CS scatter: buried vs exposed coil, per protein, mean psi(i) variance vs median |error| =================
# Run after the cross-model cell (reuses A, MIN_RES, B_BOOT, FIG, set_style). One panel per exposure, shared y-axis;
# each panel has its own per-protein Spearman rho and protein-bootstrap 95% CI.
from scipy.stats import spearmanr, pearsonr

def set_style(base=13, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base, "axes.labelsize": base,
        "xtick.labelsize": base - 3, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",
    })
set_style(base=18)
XCOL, MODEL = "psi_var_self", "SLAE-CS"
PANEL_EXP = ["buried", "exposed"]

fig, axs = plt.subplots(1, 2, figsize=(10.4, 5.2), sharey=True, gridspec_kw=dict(wspace=0.08))
for ax, exp_ in zip(axs, PANEL_EXP):
    sub = A[(A.exposure == exp_) & (A.ss == "coil")][["entry", XCOL, MODEL]].dropna()
    g = sub.groupby("entry").agg(x=(XCOL, "mean"), err=(MODEL, "median"), n=(MODEL, "size"))
    g = g[g.n >= MIN_RES]
    rho, p_rho = spearmanr(g.x, g.err)
    r, p_r = pearsonr(g.x, g.err)
    rng_b = np.random.default_rng(0)
    boot = [spearmanr(*g.iloc[rng_b.integers(0, len(g), len(g))][["x", "err"]].T.values)[0] for _ in range(B_BOOT)]
    ci_lo, ci_hi = np.nanpercentile(boot, [2.5, 97.5])
    print(f"{exp_} coil: {len(g)} proteins (median {int(g.n.median())} residues each)  "
          f"Spearman rho={rho:.3f} [{ci_lo:.3f}, {ci_hi:.3f}] p={p_rho:.2g};  Pearson r={r:.3f} p={p_r:.2g}")

    ax.scatter(g.x, g.err, s=90, color="#98C379", alpha=0.8, edgecolor="white", lw=0.6, zorder=3)
    b1, b0 = np.polyfit(g.x, g.err, 1)
    xs = np.linspace(g.x.min(), g.x.max(), 50)
    ax.plot(xs, b1 * xs + b0, color="#444", lw=1.8, ls="--", zorder=2)
    # ax.text(0.97, 0.97, f"Spearman ρ = {rho:.2f}\np = {p_rho:.2g}\nn = {len(g)} proteins",
    #         transform=ax.transAxes, ha="right", va="top")
    ax.text(0.97, 0.97, f"Spearman ρ = {rho:.2f}\np = {p_rho:.2g}",
            transform=ax.transAxes, ha="right", va="top")
    ax.set_xlim(0, 0.39)
    ax.set_xlabel("mean ψ(i) circular variance")
    ax.set_title(f"{exp_.capitalize()} loop")
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)
axs[0].set_ylim(0, 2.7)
axs[0].set_ylabel(r"median |pred - exp| $\mathregular{^{15}}$N (ppm)")
fig.suptitle("SLAE-CS", y=0.97, fontsize=18, font="Arial")

name = "s2psi_psivarself_perprotein_scatter_slaecs_buried_exposed_coil_panav10seedens"
fig.savefig(f"{FIG}/{name}.svg", bbox_inches="tight")
fig.savefig(f"{FIG}/{name}.png", dpi=300, bbox_inches="tight")
print("saved", name)
plt.show()


In [ ]:
# ================= deviation of exp 15N from the BMRB average: buried vs exposed loop =================
# Run after the cross-model cell (reuses A, gt, inv, B_BOOT, FIG, set_style).
# Reference = per-residue-type BMRB average backbone N shift from makeshift (makeshift/data/bmrb_stats.csv).
# rc_dev = |exp N - BMRB average of that residue type| (database average over all secondary structures, not a
# sequence-corrected random-coil value). Box = median / IQR / 1.5 IQR whiskers; diamond = mean.
# Stars = buried - exposed difference of medians, two-sided protein-cluster bootstrap p.
BMRB_STATS = "../../external/makeshift/makeshift/data/bmrb_stats.csv"
AA3 = {"ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H",
       "ILE": "I", "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
       "TYR": "Y", "VAL": "V"}
MODEL = "SLAE-CS"                   # only used to keep the same gated residue set as the error analyses
PANEL_EXP = ["buried", "exposed"]
EXP_C = {"buried": "#4FA8A0", "exposed": "#A8DDE3"}

bm = pd.read_csv(BMRB_STATS)
bm = bm[bm.atom == "N"]
BMRB_N = dict(zip(bm.residue.map(AA3), bm["mean"]))
assert len(BMRB_N) >= 19, f"BMRB N means missing: {sorted(set(AA3.values()) - set(BMRB_N))}"


def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."


A["aa"] = [gt["sequence"][inv[e]][r - 1] for e, r in zip(A.entry, A.resnum)]
A["exp_N"] = pd.to_numeric(pd.Series([gt["N"][inv[e]][r - 1] for e, r in zip(A.entry, A.resnum)], index=A.index),
                           errors="coerce")
A["rc_dev"] = (A.exp_N - A.aa.map(BMRB_N)).abs()
L = A[(A.ss == "coil") & A[MODEL].notna() & A.rc_dev.notna()].reset_index(drop=True)
summ = L.groupby("exposure").rc_dev.agg(["size", "median", "mean"]).reindex(PANEL_EXP)
print(summ.round(2).to_string())

# protein-cluster bootstrap of the buried - exposed difference of medians
rng_rc = np.random.default_rng(0)
rows = {p: np.where(L.entry.values == p)[0] for p in L.entry.unique()}
ents = np.array(list(rows))
dev, isb = L.rc_dev.to_numpy(), (L.exposure == "buried").to_numpy()
bs = []
for _ in range(B_BOOT):
    i = np.concatenate([rows[p] for p in rng_rc.choice(ents, ents.size)])
    bs.append(np.median(dev[i][isb[i]]) - np.median(dev[i][~isb[i]]))
bs = np.array(bs)
diff = np.median(dev[isb]) - np.median(dev[~isb])
p_dev = 2 * min((bs <= 0).mean(), (bs >= 0).mean())
print(f"median buried - exposed = {diff:+.2f} ppm [{np.percentile(bs, 2.5):+.2f}, {np.percentile(bs, 97.5):+.2f}]  "
      f"bootstrap p = {p_dev:.2g} (B = {B_BOOT})")

set_style(base=16)
fig, ax = plt.subplots(figsize=(4.4, 5.2))
data_ = [L.rc_dev[L.exposure == e].to_numpy() for e in PANEL_EXP]
bp = ax.boxplot(data_, widths=0.55, showfliers=False, patch_artist=True, medianprops=dict(color="k", lw=1.6))
for patch, e in zip(bp["boxes"], PANEL_EXP):
    patch.set(facecolor=EXP_C[e], edgecolor="#444")

from matplotlib.patches import Patch
handles = [Patch(facecolor=EXP_C[e], edgecolor="#444",
                 label=f"{e.capitalize()}: median {summ.loc[e, 'median']:.2f} ppm") for e in PANEL_EXP]

top = max(np.percentile(v, 75) + 1.5 * (np.percentile(v, 75) - np.percentile(v, 25)) for v in data_)
ax.plot([1, 1, 2, 2], [top * 1.04, top * 1.07, top * 1.07, top * 1.04], color="k", lw=1)
ax.text(1.5, top * 1.08, stars(p_dev), ha="center", va="bottom")
ax.set_xticks([1, 2])
ax.set_xticklabels([f"{e.capitalize()} loop" for e in PANEL_EXP])
ax.set_ylim(0, top * 1.2)
ax.set_ylabel(r"|experiment - BMRB average| $\mathregular{^{15}}$N (ppm)")
# ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.14), frameon=False, fontsize=13,
#           handletextpad=0.3)
for s_ in ("top", "right"):
    ax.spines[s_].set_visible(False)

name = "rcdev_bmrbavg_buried_exposed_loop_slaecs_panav10seedens"
fig.savefig(f"{FIG}/{name}.svg", bbox_inches="tight")
fig.savefig(f"{FIG}/{name}.png", dpi=300, bbox_inches="tight")
print("saved", name)
plt.show()
